# Project: Wildfire Mapping

Goal: Build an html side with an interactive map where the user can see the recents wildfires. Provide the user with information of phisical size, duration, intensity, etc. of the wildfires. Use pop-ups and tooltips to make the map interactive and structured for the users. The map should be for public users which are interesting in wildfires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the wildfire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data (if needed)
5. Check wildfires for different properties
6. Visulaization of the wildfires
7. Provide additional information about the wildfires. 
8. Make the map interactive

In [47]:
# import all libraries
import requests
import pandas as pd
import io
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
from shapely.geometry import MultiPoint
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json

In [2]:
# Building the api url 

# get api key
api_key = "47778cf594ae276d2b7dfc098596de2a"
# define source
api_source = "VIIRS_SNPP_NRT" # or change ot to VIIRS_SNPP_SP?
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today
api_day_range = 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

In [3]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

else:
    print(f"Request failed. Status code: {response.status_code}")

API request successfull


In [ ]:
# converting the data data frame into a geo-data frame
wildfire_gdf_crs4326 = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]), crs=4326)

# convert the date column into a date datetime type
wildfire_gdf_crs4326["acq_date"] = pd.to_datetime(wildfire_gdf_crs4326["acq_date"], format="%Y-%m-%d")

# preject to calculate in meters
wildfire_gdf_crs3857 = wildfire_gdf_crs4326.to_crs(epsg=3857)
# join nearby spatial points according to the same fire (most likly the same)
wildfire_gdf_crs3857["geometry_buffer"] = wildfire_gdf_crs3857.geometry.buffer(1501) # buffer of 1501 meters. resolution of the data is 350mx750m
# join fires within the buffer
joined_wildfires = gpd.sjoin(
    wildfire_gdf_crs4326[["acq_date", "geometry"]],
    wildfire_gdf_crs3857[["acq_date", "geometry_buffer"]].set_geometry("geometry_buffer"),
    how="left", # make sure to keep all points
    predicate="within"
)

joined_wildfires = joined_wildfires[joined_wildfires["acq_date_left"] == joined_wildfires["acq_date_right"]] # Keep only fires with the same registration date
# aggregate into clusters
wildfire_clusters = (
    joined_wildfires.groupby("index_right").agg(
        acq_date=("acq_date_left", "first"),
        count=("acq_date_left", "count"),
        geometry=("geometry", lambda geoms: MultiPoint(list(geoms)).centroid)
    )
    .reset_index(drop=True)
)

# Build the cleaned aggregated wildfire data
wildfire_clusterd = gpd.GeoDataFrame(wildfire_clusters, geometry="geometry", crs=3857).to_crs(epsg=4326)


In [ ]:
# verify transformation to geo data frame
display(wildfire_gdf_crs4326.head(5))
display(wildfire_clusterd.head(5))
display(wildfire_clusterd.info)
display(wildfire_clusterd.dtypes)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry,geometry_buffer
0,22.70248,39.04295,307.43,0.50,0.66,2026-05-12,1,N,VIIRS,n,2.0NRT,292.96,1.04,N,POINT (4346241.313 2596078.06),"POLYGON ((4347742.313 2596078.06, 4347735.085 ..."
1,23.94810,38.26667,325.12,0.42,0.61,2026-05-12,1,N,VIIRS,n,2.0NRT,295.73,2.29,N,POINT (4259826.219 2747085.143),"POLYGON ((4261327.219 2747085.143, 4261319.991..."
2,23.95083,38.26488,327.78,0.42,0.61,2026-05-12,1,N,VIIRS,n,2.0NRT,295.62,2.88,N,POINT (4259626.957 2747417.674),"POLYGON ((4261127.957 2747417.674, 4261120.729..."
3,24.27637,37.56365,307.05,0.38,0.58,2026-05-12,1,N,VIIRS,n,2.0NRT,295.67,0.95,N,POINT (4181566.39 2787121.342),"POLYGON ((4183067.39 2787121.342, 4183060.163 ..."
4,24.95234,32.91698,337.41,0.39,0.44,2026-05-12,1,N,VIIRS,n,2.0NRT,292.75,4.46,N,POINT (3664301.452 2869891.802),"POLYGON ((3665802.452 2869891.802, 3665795.224..."


,acq_date,count,geometry
0,2026-05-12,1,POINT (39.04295 22.70248)
1,2026-05-12,3,POINT (38.26604 23.94927)
2,2026-05-12,3,POINT (38.26604 23.94927)
3,2026-05-12,2,POINT (37.56424 24.276)
4,2026-05-12,2,POINT (32.91509 24.95271)


<bound method DataFrame.info of          acq_date  count                    geometry
0      2026-05-12      1   POINT (39.04295 22.70248)
1      2026-05-12      3   POINT (38.26604 23.94927)
2      2026-05-12      3   POINT (38.26604 23.94927)
3      2026-05-12      2     POINT (37.56424 24.276)
4      2026-05-12      2   POINT (32.91509 24.95271)
...           ...    ...                         ...
133015 2026-05-16      2  POINT (-93.46958 16.79983)
133016 2026-05-16      1  POINT (-95.06914 16.80403)
133017 2026-05-16      1   POINT (-93.4585 16.81378)
133018 2026-05-16      1  POINT (-92.28959 16.92354)
133019 2026-05-16      1   POINT (-92.2988 16.94141)

[133020 rows x 3 columns]>

acq_date    datetime64[us]
count                int64
geometry          geometry
dtype: object

In [7]:
# initalize Folium back ground map
background_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter", # Dark basmap
    control_scale=True # Add scalebar
)

cluster_fire = MarkerCluster(name="Recent Fires").add_to(background_map)
# build markers for the wildfires
for idx, row in wildfire_clusterd.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # eextract longitude out of the geometry column
    count = row["count"] # counting how many fires are aggregated

    # Formating the Tooltip. Date of the fire registation
    fire_start = row["acq_date"].date()
    tooltip = f"Date: {fire_start} | Detections: {count}"

    # Define Marker color deoending the count of fires
    if count == 1:
        color =  "orange"
    elif count <= 5:
        color = "red"
    else:
        color = "darkred"

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip,
        icon=folium.Icon(color=color, icon="fire", prefix="fa")
    ).add_to(cluster_fire)

folium.LayerControl().add_to(background_map)
background_map.save("../outputs/map.html")

In addition map I also decided to give some Insighs about in which region most open fires are recorded

In [61]:
# load Geo Data frame of the world countries
world_import = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip").to_crs(epsg=4326) # load Countries from online source.

# inspect worl gdf
display(world_import.head(5))
print(world_import.columns.tolist())

# cleaning the world 
world = world_import[["NAME", "ISO_A3", "CONTINENT", "SUBREGION", "geometry"]] # just keep usefull attributes ot of the world gdf.
display(world)

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,NaN,NaN,Unrecognized,NaN,NaN,NaN,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_VN', 'ADM0_A3_TR', 'AD

,NAME,ISO_A3,CONTINENT,SUBREGION,geometry
0,Fiji,FJI,Oceania,Melanesia,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Tanzania,TZA,Africa,Eastern Africa,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,W. Sahara,ESH,Africa,Northern Africa,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Canada,CAN,North America,Northern America,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,United States of America,USA,North America,Northern America,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."
...,...,...,...,...,...
172,Serbia,SRB,Europe,Southern Europe,"POLYGON ((18.82982 45.90887, 18.82984 45.90888..."
173,Montenegro,MNE,Europe,Southern Europe,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,..."
174,Kosovo,-99,Europe,Southern Europe,"POLYGON ((20.59025 41.85541, 20.52295 42.21787..."
175,Trinidad and Tobago,TTO,North America,Caribbean,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895..."


In [50]:
# calculate fires in the diffrent countries
# here the wildfire cluster is used, so nearby fires are counted as one fire and not as several

wildfire_countries = gpd.sjoin(wildfire_clusterd, world, 
                                how="left", predicate="within") # assign to each fire in which country it is located

# Check for fires not located inside a country for example this are burning ships, fires in artica or antarctica
unmatched = wildfire_countries[wildfire_countries["NAME"].isna()]
print(f"Unmatched fires: {len(unmatched)}")
display(wildfire_countries.head(5))

Unmatched fires: 2650


,acq_date,count,geometry,index_right,NAME,ISO_A3,CONTINENT,SUBREGION
0,2026-05-12,1,POINT (39.04295 22.70248),158.0,Saudi Arabia,SAU,Asia,Western Asia
1,2026-05-12,3,POINT (38.26604 23.94927),158.0,Saudi Arabia,SAU,Asia,Western Asia
2,2026-05-12,3,POINT (38.26604 23.94927),158.0,Saudi Arabia,SAU,Asia,Western Asia
3,2026-05-12,2,POINT (37.56424 24.276),158.0,Saudi Arabia,SAU,Asia,Western Asia
4,2026-05-12,2,POINT (32.91509 24.95271),163.0,Egypt,EGY,Africa,Northern Africa


In [59]:
# count the fires for each country
fire_per_country = world[["NAME", "CONTINENT","ISO_A3", "geometry"]].copy() # copy world to build fire per country
# merge fire counts in — countries with no fires get NaN
fire_per_country = fire_per_country.merge(
    wildfire_countries.groupby("NAME").size().reset_index(name="fire_count"),
    on="NAME",
    how="left"  # keep all countries from world
)

fire_per_country["fire_count"] = fire_per_country["fire_count"].fillna(0).astype(int) # fill the NaN values of the countries without fires with 0

fire_per_country = gpd.GeoDataFrame(fire_per_country, geometry="geometry", crs=4326) # 

display(fire_per_country)

,NAME,CONTINENT,ISO_A3,geometry,fire_count
0,Fiji,Oceania,FJI,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...",0
1,Tanzania,Africa,TZA,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...",330
2,W. Sahara,Africa,ESH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...",0
3,Canada,North America,CAN,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...",443
4,United States of America,North America,USA,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...",6337
...,...,...,...,...,...
172,Serbia,Europe,SRB,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...",12
173,Montenegro,Europe,MNE,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...",0
174,Kosovo,Europe,-99,"POLYGON ((20.59025 41.85541, 20.52295 42.21787...",0
175,Trinidad and Tobago,North America,TTO,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...",33


In [ ]:
# filter country with no  fire
no_fire = fire_per_country[fire_per_country["fire_count"] == 0]


# make a map displaying the nubers of fire per each country
fig = px.choropleth(
    fire_per_country,
    locations="ISO_A3",
    color="fire_count",
    hover_name="NAME",
    custom_data=["fire_count"],
    color_continuous_scale="OrRd"
)
# costume hover template
fig.update_traces(hovertemplate="<b>%{hovertext}</b><br>Fires recorded: %{customdata[0]}<extra></extra>")

fig.update_layout(
    # title
    title=dict(
        text="Fires per Country",
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),

    # legend/colorbar
    coloraxis_colorbar=dict(
        title="Fires recorded",
        thickness=15,
        len=0.5,
    ),

    # grid coordinates on map frame
    geo=dict(
        showframe=True,
        framecolor="grey",
        showcoastlines=True,
        coastlinecolor="grey",
        showland=True,
        landcolor="lightgrey",
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    ),
)

fig.write_html("../outputs/fire_per_country.html")